# Классификация типов опухолей по данным RNA-Seq

## 1. Постановка задачи

Задача проекта — классифицировать пять типов опухолей (BRCA, COAD, KIRC, LUAD и PRAD) по профилям RNA-Seq. Набор содержит 801 объект и 20 531 исходный признак. Проект выполнен в учебных целях, не имеет клинического назначения и не является медицинской диагностической системой.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "results").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "results").is_dir():
    raise FileNotFoundError("Запустите notebook из корня репозитория или папки notebooks")
pd.set_option("display.max_columns", None)

## 2. Обзор данных

Сводка ниже читается из сохранённых JSON-артефактов. Исходный архив датасета не загружается.

![Распределение классов](../figures/class_distribution.png)

In [2]:
dataset_summary = json.loads(
    (PROJECT_ROOT / "results/dataset_summary.json").read_text(encoding="utf-8")
)
split_summary = json.loads(
    (PROJECT_ROOT / "results/split_summary.json").read_text(encoding="utf-8")
)
overview = pd.Series(
    {
        "Объекты": dataset_summary["sample_count"],
        "Исходные признаки": dataset_summary["feature_count"],
        "Классы": dataset_summary["class_count"],
        "Train": split_summary["train_samples"],
        "Test": split_summary["test_samples"],
    },
    name="Значение",
).to_frame()
display(overview)
display(pd.Series(dataset_summary["class_distribution"], name="Количество").to_frame())
display(pd.Series(dataset_summary["quality"], name="Количество").to_frame())

,Значение
Объекты,801
Исходные признаки,20531
Классы,5
Train,640
Test,161


,Количество
BRCA,300
COAD,78
KIRC,146
LUAD,141
PRAD,136


,Количество
missing_values,0
infinite_values,0
duplicate_sample_ids,0
duplicate_feature_names,0
duplicate_feature_rows,0
constant_features,267


## 3. Экспериментальный протокол

Данные были разделены стратифицированно в пропорции 80/20 с `random_state=42`. Test set оставался закрытым во время сравнения конфигураций. Все исследовательские оценки получены с помощью 5-fold cross-validation только на train-части. Preprocessing, SelectKBest и PCA обучались отдельно внутри каждого training fold, что предотвращало data leakage. Основная метрика — macro F1, одинаково учитывающая пять классов. E10 был зафиксирован до единственного открытия test set.

## 4. Сравнение экспериментов

Таблица содержит все E00–E17: baseline, SelectKBest, PCA, Random Forest и PyTorch MLP. `n_features` означает число исходных признаков для baseline, число выбранных признаков для SelectKBest или число компонент для PCA.

In [3]:
metrics = pd.read_csv(PROJECT_ROOT / "results/metrics.csv")
display(metrics)

,experiment_id,model,feature_method,n_features,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,cv_precision_macro_mean,cv_precision_macro_std,cv_recall_macro_mean,cv_recall_macro_std,cv_fit_time_mean,cv_score_time_mean,total_cv_time_seconds,random_state,cv_splits
0,E00,DummyClassifier,none,20531,0.109091,0.000000,0.375000,0.000000,0.075000,0.000000,0.200000,0.000000,0.156331,0.002744,0.834388,42,5
1,E01,LogisticRegression,none,20531,0.995388,0.006527,0.995313,0.006250,0.993517,0.009372,0.997500,0.003333,0.379071,0.070927,2.290569,42,5
2,E02,LogisticRegression,select_k_best_f_classif,50,0.997356,0.003239,0.996875,0.003827,0.998367,0.002000,0.996443,0.004359,0.325343,0.054146,1.938009,42,5
3,E03,LogisticRegression,select_k_best_f_classif,100,0.996055,0.003222,0.995313,0.003827,0.997551,0.002000,0.994704,0.004327,0.308850,0.048866,1.827299,42,5
4,E04,LogisticRegression,select_k_best_f_classif,200,0.997397,0.003187,0.996875,0.003827,0.998367,0.002000,0.996522,0.004260,0.309644,0.048808,1.830760,42,5
5,E05,LogisticRegression,select_k_best_f_classif,500,0.998657,0.002685,0.998437,0.003125,0.999184,0.001633,0.998182,0.003636,0.314154,0.048537,1.851533,42,5
6,E06,LinearSVC,select_k_best_f_classif,50,0.998699,0.002603,0.998437,0.003125,0.999184,0.001633,0.998261,0.003478,0.289828,0.061337,1.793325,42,5
7,E07,LinearSVC,select_k_best_f_classif,100,0.998699,0.002603,0.998437,0.003125,0.999184,0.001633,0.998261,0.003478,0.322796,0.047554,1.889950,42,5
8,E08,LinearSVC,select_k_best_f_classif,200,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.398403,0.047040,2.265857,42,5
9,E09,LinearSVC,select_k_best_f_classif,500,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.129925,0.060419,5.989566,42,5


In [4]:
representative_ids = ["E00", "E01", "E08", "E10", "E16", "E17"]
compact_columns = [
    "experiment_id",
    "model",
    "feature_method",
    "n_features",
    "cv_f1_macro_mean",
    "cv_f1_macro_std",
    "total_cv_time_seconds",
]
compact_comparison = metrics.loc[
    metrics["experiment_id"].isin(representative_ids), compact_columns
].copy()
display(compact_comparison)

,experiment_id,model,feature_method,n_features,cv_f1_macro_mean,cv_f1_macro_std,total_cv_time_seconds
0,E00,DummyClassifier,none,20531,0.109091,0.000000,0.834388
1,E01,LogisticRegression,none,20531,0.995388,0.006527,2.290569
8,E08,LinearSVC,select_k_best_f_classif,200,1.000000,0.000000,2.265857
10,E10,LogisticRegression,pca,20,1.000000,0.000000,2.332715
16,E16,RandomForestClassifier,select_k_best_f_classif,200,0.996055,0.003222,2.830478
17,E17,PyTorchMLP,select_k_best_f_classif,200,0.997356,0.003239,3.996164


Графики были построены на исследовательских этапах и здесь только отображаются. Двадцать компонент E10 — плотное внутреннее представление, рассчитанное по всем 20 531 исходным признакам, а не набор из 20 генов.

![Сравнение SelectKBest](../figures/feature_selection_comparison.png)

![Сравнение PCA](../figures/pca_comparison.png)

![Сравнение моделей](../figures/model_comparison.png)

![PCA-проекция train](../figures/pca_train_projection.png)

## 5. Выбор кандидата

Сохранённый манифест кандидата фиксирует E10, точный Pipeline, источник CV-результатов, tie-break и контрольные суммы split до обращения к test set.

In [5]:
candidate = json.loads((PROJECT_ROOT / "results/final_candidate.json").read_text(encoding="utf-8"))
candidate_summary = pd.Series(
    {
        "Статус": candidate["selection_status"],
        "Эксперимент": candidate["selected_experiment_id"],
        "Модель": candidate["selected_model"],
        "Метод": candidate["feature_method"],
        "Компоненты": candidate["representation_dimensions"],
        "CV macro F1": candidate["selection_metric_value"],
        "Commit источника": candidate["selection_source_commit"],
    },
    name="Значение",
).to_frame()
pipeline_table = pd.DataFrame.from_dict(candidate["pipeline"], orient="index")
split_checksums = pd.Series(
    {
        "Train index SHA-256": candidate["split"]["train_index_sha256"],
        "Test index SHA-256": candidate["split"]["test_index_sha256"],
    },
    name="Значение",
).to_frame()
display(candidate_summary)
display(pipeline_table)
display(pd.Series(candidate["tie_break_rule"], name="Tie-break").to_frame())
display(split_checksums)

,Значение
Статус,frozen_before_test
Эксперимент,E10
Модель,LogisticRegression
Метод,pca
Компоненты,20
CV macro F1,1.0
Commit источника,0303133aef4ab853c30e7ee142f84067b36c7c98


,class,n_components,svd_solver,random_state,whiten,max_iter
scaler,StandardScaler,NaN,NaN,NaN,NaN,NaN
pca,PCA,20.0,randomized,42.0,False,NaN
classifier,LogisticRegression,NaN,NaN,42.0,NaN,3000.0


,Tie-break
0,maximum cv_f1_macro_mean
1,smaller internal representation
2,lower execution time
3,simpler model


,Значение
Train index SHA-256,5b6a7488ce4e227431d25515675bcb8ad544fcf65620e0...
Test index SHA-256,833cf2ee6a93f35eb303125d093170889a808a62014a0e...


## 6. Финальная оценка

Ниже отображаются уже сохранённые результаты единственной финальной оценки E10. Повторное обучение и повторная оценка не выполняются.

![Финальная confusion matrix](../figures/final_confusion_matrix.png)

In [6]:
final_evaluation = json.loads(
    (PROJECT_ROOT / "results/final_evaluation.json").read_text(encoding="utf-8")
)
final_metrics = pd.Series(final_evaluation["test_metrics"], name="Test").to_frame()
cv_test_comparison = pd.DataFrame(
    {
        "Macro F1": [
            final_evaluation["cv_reference"]["cv_f1_macro_mean"],
            final_evaluation["test_metrics"]["f1_macro"],
        ]
    },
    index=["Train CV", "Test"],
)
display(final_metrics)
display(cv_test_comparison)

,Test
f1_macro,1.0
accuracy,1.0
precision_macro,1.0
recall_macro,1.0


,Macro F1
Train CV,1.0
Test,1.0


In [7]:
classification_report = json.loads(
    (PROJECT_ROOT / "results/final_classification_report.json").read_text(encoding="utf-8")
)
display(pd.DataFrame(classification_report).T)

,precision,recall,f1-score,support
BRCA,1.0,1.0,1.0,60.0
COAD,1.0,1.0,1.0,16.0
KIRC,1.0,1.0,1.0,30.0
LUAD,1.0,1.0,1.0,28.0
PRAD,1.0,1.0,1.0,27.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,161.0
weighted avg,1.0,1.0,1.0,161.0


In [8]:
confusion_matrix = pd.read_csv(PROJECT_ROOT / "results/final_confusion_matrix.csv")
confusion_counts = confusion_matrix.set_index("true_label")
class_order = final_evaluation["class_order"]
correct_predictions = sum(int(confusion_counts.loc[label, label]) for label in class_order)
total_predictions = int(confusion_counts.to_numpy().sum())
prediction_counts = pd.Series(
    {
        "Правильные": correct_predictions,
        "Ошибочные": total_predictions - correct_predictions,
    },
    name="Количество",
).to_frame()
display(confusion_counts)
display(prediction_counts)

,BRCA,COAD,KIRC,LUAD,PRAD
true_label,,,,,
BRCA,60,0,0,0,0
COAD,0,16,0,0,0
KIRC,0,0,30,0,0
LUAD,0,0,0,28,0
PRAD,0,0,0,0,27


,Количество
Правильные,161
Ошибочные,0


In [9]:
timings = pd.Series(
    {
        "Обучение, с": final_evaluation["training_time_seconds"],
        "Predict, с": final_evaluation["prediction_time_seconds"],
        "Predict proba, с": final_evaluation["probability_prediction_time_seconds"],
    },
    name="Время",
).to_frame()
runtime = pd.Series(final_evaluation["runtime"], name="Версия").to_frame()
model_artifact = pd.Series(final_evaluation["artifacts"]["model"], name="Модель").to_frame()
display(timings)
display(runtime)
display(model_artifact)

,Время
"Обучение, с",0.252369
"Predict, с",0.068052
"Predict proba, с",0.056737


,Версия
python,3.12.13
numpy,2.4.6
pandas,3.0.3
scikit_learn,1.9.0
joblib,1.5.3


,Модель
path,models/final_e10_pipeline.joblib
sha256,8082bfa3223209971847064053254d9e5c52a2d758c338...


## 7. Выводы и ограничения

Все 161 test-объекта классифицированы правильно. Итоговый Pipeline использует 20 PCA-компонент, построенных по всем 20 531 исходным признакам. Результат относится только к этому публичному набору и одному заранее сформированному holdout split. Независимая когорта и влияние batch effects не исследовались. Условные имена `gene_*` не позволяют делать выводы о конкретных биомаркерах. Проект является учебным и не предназначен для медицинской диагностики.